In [ ]:
from pathlib import Path
import os, sys
if (Path.cwd() / "AGENTS.md").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "AGENTS.md").exists():
    ROOT = Path.cwd().parent
else:
    ROOT = Path.home() / "hibah-riset"
DATA, EXT, EXP = ROOT / "data" / "s2", ROOT / "external", ROOT / "experiments" / "s2_tracker"
for d in (DATA, EXP):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(S2_ROOT=str(ROOT), S2_DATA=str(DATA), S2_EXT=str(EXT), S2_EXP=str(EXP))
print("ROOT :", ROOT)
print("DATA :", DATA)
print("python:", sys.executable)

# 50 — Jalankan Tracking DiffMOT (MOT20 + DanceTrack)

**Kernel: `s2-diffmot`**.

Prasyarat: notebook 10 (env+bobot), 20 (data), 30 (deteksi), 40 (patch+smoke).

Config ditulis SENDIRI (config `*_test.yaml` di repo DiffMOT stale — semua copy-paste path
`/mnt/8T/...` DanceTrack). Threshold rilis: MOT20 high 0.4/low 0.1; DanceTrack 0.6/0.4;
`w_assoc_emb=2.2`, `aw_param=1.7`.

Checkpoint motion dimuat dari `{diffmot}/experiments/{eval_expname}/{dataset}_epoch800.pt`
→ notebook 10 sudah menaruh `mot_epoch800.pt` dan `dancetrack_epoch800.pt` di posisi yang benar.

In [ ]:
from pathlib import Path
import os, sys
if (Path.cwd() / "AGENTS.md").exists():
    ROOT = Path.cwd()
elif (Path.cwd().parent / "AGENTS.md").exists():
    ROOT = Path.cwd().parent
else:
    ROOT = Path.home() / "hibah-riset"
DATA, EXT, EXP = ROOT / "data" / "s2", ROOT / "external", ROOT / "experiments" / "s2_tracker"
for d in (DATA, EXP):
    d.mkdir(parents=True, exist_ok=True)
os.environ.update(S2_ROOT=str(ROOT), S2_DATA=str(DATA), S2_EXT=str(EXT), S2_EXP=str(EXP))
print("ROOT :", ROOT)
print("DATA :", DATA)
print("python:", sys.executable)

### Tulis config

In [ ]:
import yaml, sys
sys.path.insert(0, str(EXT / "diffmot"))

def make_config(ds, split, eval_expname, high_thres, low_thres):
    return {
        "eps": 0.001, "eval_mode": True, "lr": 0.0001,
        "data_dir": str(DATA / ds / split),          # aman untuk eval (guard isdir di DiffMOTDataset)
        "diffnet": "HMINet", "interval": 5, "augment": True,
        "encoder_dim": 256, "tf_layer": 3, "epochs": 800,
        "batch_size": 2048, "seed": 123, "eval_every": 20, "gpus": [0],
        "eval_at": 800,
        "det_dir": str(DATA / ds / "detections" / split),
        "info_dir": str(DATA / ds / split),          # butuh {seq}/seqinfo.ini (notebook 20)
        "reid_dir": str(DATA / "embeddings" / ds),
        "save_dir": str(EXP / "diffmot_results" / ds),
        "eval_expname": eval_expname,
        "high_thres": high_thres, "low_thres": low_thres,
        "w_assoc_emb": 2.2, "aw_param": 1.7,
        "preprocess_workers": 8, "device": "cuda", "eval_device": None,
    }

cfg_mot20 = make_config("mot20", "train", "diffmot_mot", 0.4, 0.1)
cfg_dance = make_config("dancetrack", "val", "diffmot_dance", 0.6, 0.4)
cfg_dir = EXT / "diffmot" / "configs_s2"; cfg_dir.mkdir(exist_ok=True)
with open(cfg_dir / "mot20_test.yaml", "w") as f: yaml.safe_dump(cfg_mot20, f)
with open(cfg_dir / "dancetrack_test.yaml", "w") as f: yaml.safe_dump(cfg_dance, f)
print("configs ditulis ke", cfg_dir)

### Pra-cek: checkpoint & deteksi

In [ ]:
from pathlib import Path
checks = [
    (EXT/"diffmot"/"experiments"/"diffmot_mot"/"mot_epoch800.pt", "checkpoint MOT"),
    (EXT/"diffmot"/"experiments"/"diffmot_dance"/"dancetrack_epoch800.pt", "checkpoint DanceTrack"),
    (DATA/"mot20"/"detections"/"train"/"MOT20-01"/"00000001.txt", "deteksi MOT20"),
    (DATA/"dancetrack"/"detections"/"val", "deteksi DanceTrack"),
]
ok = True
for p, label in checks:
    good = p.exists() if p.suffix else p.is_dir()
    print(("OK  " if good else "FAIL"), label, p)
    ok = ok and good
assert ok, "ada prasyarat belum lengkap"

### Run MOT20 (4 sekuens train)

Estimasi 7–10 menit di 4090. `--dataset mot` → memuat `mot_epoch800.pt`.

In [ ]:
!cd $S2_EXT/diffmot && time python main.py --config configs_s2/mot20_test.yaml --dataset mot

### Run DanceTrack (25 sekuens val)

Estimasi 15–20 menit di 4090. `--dataset dancetrack` → memuat `dancetrack_epoch800.pt`.

In [ ]:
!cd $S2_EXT/diffmot && time python main.py --config configs_s2/dancetrack_test.yaml --dataset dancetrack

### Verifikasi output + runtime

In [ ]:
from pathlib import Path
for ds in ["mot20", "dancetrack"]:
    out = EXP / "diffmot_results" / ds
    files = sorted(out.glob("*.txt")) if out.exists() else []
    print(f"{ds}: {len(files)} sekuens hasil")
    for f in files[:5]:
        n = sum(1 for _ in f.open())
        print("  ", f.name, n, "baris")

**Lanjut**: `60_s2_run_ocsort.ipynb` (kernel `s2-main`).